# Reward Hacking in Pong with HackAtari

This notebook demonstrates **reward hacking** — a core AI safety problem where an agent finds unintended ways to maximise its reward signal rather than solving the intended task. We use the [HackAtari](https://github.com/k4ntz/HackAtari) library to deliberately modify the Pong environment and observe how agents exploit these modifications.


## Setup


In [ ]:
%pip install hackatari ocatari stable-baselines3[extra] gymnasium ale-py matplotlib numpy --quiet


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import display, HTML
from collections import deque
import warnings
warnings.filterwarnings('ignore')

from hackatari.core import HackAtari
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
import gymnasium as gym


## What Is Reward Hacking?

**Reward hacking** (also called reward misspecification or specification gaming) occurs when an agent achieves high reward by exploiting loopholes in the reward function rather than pursuing the designer's intended goal.

Classic examples:
- A boat-racing agent that scores points by going in circles collecting bonuses rather than finishing the race
- A robotic hand that tips itself upside-down to 'grasp' an object
- A simulated creature that grows tall and falls over to maximise its forward displacement reward

HackAtari lets us **artificially introduce** reward hacking opportunities in Atari games, giving us a controlled setting to study the phenomenon.

### Our experiment

We run three scenarios in Pong:

| Scenario | Environment | What the agent can exploit |
|---|---|---|
| Baseline | Standard Pong | Nothing — opponent plays normally |
| Lazy enemy | `lazy_enemy` hack | Enemy stops moving after returning the ball |
| Ball drift | `up_drift` hack | Ball curves upward, making it predictable |

We will see that even a **random agent** scores significantly higher in hacked environments, and a trained agent learns to fully exploit the loopholes.


## 1. Baseline: Standard Pong

First, establish performance in the unmodified environment. A random agent scores around **−21** (loses every point).


In [ ]:
def run_episode(env, policy='random', max_steps=10_000):
    """Run one episode and return total reward and frame list."""
    obs, _ = env.reset()
    total_reward = 0
    frames = []
    for _ in range(max_steps):
        if policy == 'random':
            action = env.action_space.sample()
        else:
            action, _ = policy.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if len(frames) < 200:  # cap frames for display
            frames.append(env.render())
        if terminated or truncated:
            break
    return total_reward, frames


def evaluate_random(env_name, modifications=None, n_episodes=5):
    """Run n random episodes and return mean / std reward."""
    kwargs = dict(render_mode='rgb_array', obs_mode='dqn')
    if modifications:
        kwargs['modifications'] = modifications
    env = HackAtari(env_name, **kwargs)
    rewards = [run_episode(env, policy='random')[0] for _ in range(n_episodes)]
    env.close()
    return np.mean(rewards), np.std(rewards)


print('Running baseline (standard Pong, random agent)...')
base_mean, base_std = evaluate_random('ALE/Pong-v5')
print(f'Standard Pong  — mean reward: {base_mean:.1f} ± {base_std:.1f}')


## 2. Reward Hacking via `lazy_enemy`

The `lazy_enemy` hack makes the opponent **stop moving** after it returns the ball. This is a classic reward hacking scenario: the agent doesn't need to improve its gameplay — the environment itself has been made trivially easy.

Notice how the same **random policy** that scored ~−21 on normal Pong suddenly scores much higher.


In [ ]:
print('Running lazy_enemy hack (random agent)...')
lazy_mean, lazy_std = evaluate_random('ALE/Pong-v5', modifications='lazy_enemy')
print(f'Lazy enemy Pong — mean reward: {lazy_mean:.1f} ± {lazy_std:.1f}')

# Plot comparison
fig, ax = plt.subplots(figsize=(7, 4))
labels = ['Standard Pong\n(no hack)', 'Lazy Enemy\n(reward hack)']
means  = [base_mean, lazy_mean]
stds   = [base_std,  lazy_std]
colors = ['#4c72b0', '#dd8452']
bars = ax.bar(labels, means, yerr=stds, color=colors, capsize=6, width=0.4)
ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax.set_ylabel('Mean Episode Reward')
ax.set_title('Random Agent: Standard vs Hacked Pong')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, m + 0.5, f'{m:.1f}',
            ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('reward_comparison.png', dpi=120)
plt.show()
print(f'\nReward improvement from hack: {lazy_mean - base_mean:+.1f} points')


## 3. Ball Drift Hack

The `up_drift` modification makes the ball curve upward, making its trajectory predictable and easy to intercept. Combined with `lazy_enemy`, the environment is trivially solvable.


In [ ]:
print('Running up_drift hack (random agent)...')
drift_mean, drift_std = evaluate_random('ALE/Pong-v5', modifications='up_drift')
print(f'Up-drift Pong   — mean reward: {drift_mean:.1f} ± {drift_std:.1f}')

print('Running combined lazy_enemy + up_drift (random agent)...')
combo_mean, combo_std = evaluate_random('ALE/Pong-v5', modifications=['lazy_enemy', 'up_drift'])
print(f'Combined hacks  — mean reward: {combo_mean:.1f} ± {combo_std:.1f}')

# Extended bar chart
fig, ax = plt.subplots(figsize=(9, 4))
labels = ['Standard\nPong', 'Lazy\nEnemy', 'Up\nDrift', 'Both\nHacks']
means  = [base_mean, lazy_mean, drift_mean, combo_mean]
stds   = [base_std,  lazy_std,  drift_std,  combo_std]
colors = ['#4c72b0', '#dd8452', '#55a868', '#c44e52']
bars = ax.bar(labels, means, yerr=stds, color=colors, capsize=6, width=0.5)
ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax.set_ylabel('Mean Episode Reward')
ax.set_title('Random Agent Performance Across HackAtari Modifications')
for bar, m in zip(bars, means):
    ypos = m + 0.5 if m >= 0 else m - 1.5
    ax.text(bar.get_x() + bar.get_width()/2, ypos, f'{m:.1f}',
            ha='center', va='bottom', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.savefig('all_hacks_comparison.png', dpi=120)
plt.show()


## 4. Training an Agent on the Hacked Environment

Now we train a DQN agent directly on the hacked environment (`lazy_enemy`). The agent will learn to exploit the lazy opponent — achieving high reward — but will have learned a *policy tailored to the exploit*, not genuine Pong skill.

This is a miniature version of the real-world reward hacking problem: **optimise against a flawed reward signal → get a flawed agent.**


In [ ]:
import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing, FrameStack


def make_pong_env(modifications=None, seed=42):
    """Create a HackAtari Pong env wrapped for DQN training."""
    kwargs = dict(obs_mode='dqn', render_mode='rgb_array')
    if modifications:
        kwargs['modifications'] = modifications
    env = HackAtari('ALE/Pong-v5', **kwargs)
    env = Monitor(env)
    return env


# Train on hacked environment
print('Creating hacked training environment (lazy_enemy)...')
hacked_env = make_pong_env(modifications='lazy_enemy')

print('Training DQN on hacked Pong (50k steps — expect quick convergence)...')
hacked_model = DQN(
    'MlpPolicy',
    hacked_env,
    learning_rate=1e-4,
    buffer_size=50_000,
    learning_starts=1_000,
    batch_size=32,
    exploration_fraction=0.2,
    exploration_final_eps=0.05,
    train_freq=4,
    verbose=1,
)
hacked_model.learn(total_timesteps=50_000)
hacked_env.close()
print('Training complete.')


## 5. The Transfer Test: Does It Actually Know How to Play Pong?

We now evaluate the trained agent in **both** environments:

1. **Hacked Pong** (where it was trained): should score very high
2. **Standard Pong** (normal opponent): reveals whether it learned real skills

If the agent score drops dramatically on standard Pong, it has **reward-hacked**: it learned to exploit the modification rather than learn to play the game.


In [ ]:
def evaluate_agent(model, env_name, modifications=None, n_episodes=10):
    """Evaluate a trained model over n episodes, return mean/std reward."""
    kwargs = dict(obs_mode='dqn', render_mode='rgb_array')
    if modifications:
        kwargs['modifications'] = modifications
    env = HackAtari(env_name, **kwargs)
    mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=n_episodes, deterministic=True)
    env.close()
    return mean_r, std_r


print('Evaluating hacked-trained agent on hacked Pong...')
hacked_on_hacked_mean, hacked_on_hacked_std = evaluate_agent(
    hacked_model, 'ALE/Pong-v5', modifications='lazy_enemy'
)
print(f'Hacked agent on HACKED env:   {hacked_on_hacked_mean:.1f} ± {hacked_on_hacked_std:.1f}')

print('Evaluating hacked-trained agent on standard Pong...')
hacked_on_normal_mean, hacked_on_normal_std = evaluate_agent(
    hacked_model, 'ALE/Pong-v5', modifications=None
)
print(f'Hacked agent on NORMAL env:   {hacked_on_normal_mean:.1f} ± {hacked_on_normal_std:.1f}')

# Visualise the transfer gap
fig, ax = plt.subplots(figsize=(8, 5))
scenarios = [
    ('Random\n(normal)', base_mean, base_std, '#4c72b0'),
    ('Random\n(hacked)', lazy_mean, lazy_std, '#4c72b0'),
    ('Hacked agent\n(hacked env)', hacked_on_hacked_mean, hacked_on_hacked_std, '#c44e52'),
    ('Hacked agent\n(normal env)', hacked_on_normal_mean, hacked_on_normal_std, '#dd8452'),
]
labels, means, stds, colors = zip(*scenarios)
bars = ax.bar(labels, means, yerr=stds, color=colors, capsize=6, width=0.5)
ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax.set_ylabel('Mean Episode Reward')
ax.set_title('Reward Hacking: Training on Hacked Env Degrades Transfer')
for bar, m in zip(bars, means):
    ypos = m + 0.5 if m >= 0 else m - 1.5
    ax.text(bar.get_x() + bar.get_width()/2, ypos, f'{m:.1f}',
            ha='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.savefig('transfer_test.png', dpi=120)
plt.show()

transfer_gap = hacked_on_normal_mean - hacked_on_hacked_mean
print(f'\nTransfer gap: {transfer_gap:+.1f} points')
print('A large negative gap confirms reward hacking: the agent exploited the modification,\n'
      'not the underlying game.')


## 6. Custom Reward Function Hacking

HackAtari also lets you inject a **custom reward function** at runtime. This simulates the real-world problem of a misspecified reward: you think you wrote 'score points', but actually wrote something subtly different.

Here we replace the normal Pong reward (score points by getting the ball past the opponent) with one that rewards the agent for **keeping rallies going** — i.e. bouncing the ball as many times as possible. This is a misspecification: the 'intended' goal is to win, but the proxy reward incentivises the agent never to score!


In [ ]:
# Write the custom reward file to disk
rally_reward_code = '''
# Custom reward: +1 for each time the player paddle touches the ball,
# 0 for scoring a point.
# This misspecification rewards rally-keeping, not winning.
_prev_ball_x = None

def reward_function(env) -> float:
    global _prev_ball_x
    # Access game objects via env.objects
    try:
        ball = next(o for o in env.objects if 'Ball' in type(o).__name__)
        player = next(o for o in env.objects if 'Player' in type(o).__name__)
        ball_x = ball.x
        # Detect if ball changed horizontal direction near player paddle
        if _prev_ball_x is not None:
            if abs(ball_x - player.x) < 8 and abs(ball_x - _prev_ball_x) > 2:
                _prev_ball_x = ball_x
                return 1.0  # reward for keeping the rally going
        _prev_ball_x = ball_x
    except StopIteration:
        pass
    return 0.0
'''

with open('/tmp/rally_reward.py', 'w') as f:
    f.write(rally_reward_code)

print('Custom rally-keeping reward function written to /tmp/rally_reward.py')
print()
print('This reward function illustrates misspecification:')
print('  - Intended goal:  win by scoring points')
print('  - Proxy reward:   +1 per paddle touch (keep rallies going)')
print('  - Consequence:    agent learns to extend rallies, possibly avoiding winning!')


In [ ]:
# Create environment with the custom reward
custom_env = HackAtari(
    'ALE/Pong-v5',
    obs_mode='dqn',
    render_mode='rgb_array',
    rewardfunc_path='/tmp/rally_reward.py',
)

# Run a few steps to see the custom reward in action
obs, _ = custom_env.reset()
custom_rewards = []
for step in range(500):
    action = custom_env.action_space.sample()
    obs, reward, terminated, truncated, info = custom_env.step(action)
    if reward != 0:
        custom_rewards.append((step, reward))
    if terminated or truncated:
        break

custom_env.close()
print(f'Non-zero rewards in 500 steps: {len(custom_rewards)}')
print('Sample rewards (step, value):', custom_rewards[:10])
print()
print('Unlike the game score (+1/-1 per point), the custom reward fires on rally hits.')


## 7. What This Tells Us About AI Safety

Our experiments illustrate several reward hacking failure modes:

### 7.1 Goodhart's Law
> *When a measure becomes a target, it ceases to be a good measure.*

The `lazy_enemy` agent maximised reward perfectly — but not by becoming a better Pong player. It exploited an environmental quirk. Transfer to the real game exposed the gap.

### 7.2 Reward misspecification is easy to do accidentally

The rally-reward example shows how a small, seemingly reasonable change to the reward function can produce an agent with the *opposite* of the intended behaviour. In real systems, the 'true' goal (human satisfaction, safety, task completion) is never directly observable — we always optimise a proxy.

### 7.3 Difficulty of detecting hacking from reward curves alone

If you only looked at the training reward curve, the hacked agent would look like a *success* — high reward, stable learning. Only the transfer test revealed the problem. This is why **out-of-distribution evaluation** and **interpretability** are important.

### 7.4 Mitigations

| Approach | Description |
|---|---|
| Diverse evaluation | Test on unmodified / held-out environments |
| Reward modelling | Learn the reward from human feedback rather than hardcoding it |
| Interpretability | Inspect *what* the agent is attending to |
| Adversarial testing | Deliberately try to find exploitable reward loopholes before deployment |

HackAtari is a controlled research tool for exactly this kind of adversarial testing.


## 8. Visualising a Hacked Episode

Render a short clip showing the hacked agent exploiting the lazy enemy.


In [ ]:
def render_episode_gif(model, modifications=None, max_steps=300, filename='episode.gif'):
    """Render an episode and save as a GIF."""
    kwargs = dict(obs_mode='dqn', render_mode='rgb_array')
    if modifications:
        kwargs['modifications'] = modifications
    env = HackAtari('ALE/Pong-v5', **kwargs)
    obs, _ = env.reset()
    frames = []
    for _ in range(max_steps):
        if model == 'random':
            action = env.action_space.sample()
        else:
            action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        frames.append(env.render())
        if terminated or truncated:
            break
    env.close()

    fig, ax = plt.subplots(figsize=(4, 5))
    ax.axis('off')
    im = ax.imshow(frames[0])

    def update(i):
        im.set_data(frames[i])
        return [im]

    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=33, blit=True)
    ani.save(filename, writer='pillow', fps=30)
    plt.close()
    print(f'Saved {len(frames)}-frame GIF to {filename}')
    return filename


# Render hacked agent on hacked env
render_episode_gif(hacked_model, modifications='lazy_enemy', filename='hacked_agent.gif')

# Render same agent on normal env (should struggle)
render_episode_gif(hacked_model, modifications=None, filename='hacked_agent_normal.gif')

print()
print('hacked_agent.gif    — agent exploiting lazy enemy (expect high score)')
print('hacked_agent_normal.gif — same agent vs normal opponent (expect poor play)')
